# Kev-0.8B fine-tuning run (Colab or Kaggle T4)

One run of the Milestone 4 plan per session. The run commands come from
`python -m analyst.models.kev_runs plan --out kev_plan.json` (configs/train.yaml).

Before starting:
1. Runtime: T4 GPU. The T4 has no bf16, so every run uses `--dtype fp32` (kev.train has no fp16).
2. Shared Drive folder `Group4/` holds `data/` (the split files and `kev_plan.json`) and
   `checkpoints/` (one folder per run).
3. One Colab and one Kaggle account per person; do not use other accounts to extend limits.

In [ ]:
RUN_ID = "b0_s0"                      # a run id from kev_plan.json
KEV_COMMIT = "2074cc44256200bb37540103fa4572901faa10f3"
DRIVE = "/content/drive/MyDrive/Group4"   # Kaggle: an attached dataset path instead

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys; print(sys.version)

In [ ]:
from google.colab import drive  # Kaggle: skip this cell
drive.mount("/content/drive")

In [ ]:
%cd /content
!git clone -q https://github.com/jaredpalmer/kev.git && cd kev && git checkout -q $KEV_COMMIT && git log -1 --format='%H %cd'
%cd /content/kev
!pip install -q -e .

In [ ]:
import json, pathlib, shlex
plan = {r["run_id"]: r for r in json.loads(pathlib.Path(f"{DRIVE}/data/kev_plan.json").read_text())}
run = plan[RUN_ID]
data = pathlib.Path(DRIVE) / run["data"]
assert data.exists(), data
command = run["command"].replace(f"--data {run['data']}", f"--data {shlex.quote(str(data))}")
print(command)

In [ ]:
import hashlib
print("data sha256", hashlib.sha256(data.read_bytes()).hexdigest())
!{command} 2>&1 | tee /content/train_{RUN_ID}.log

In [ ]:
import shutil
out = pathlib.Path(f"/content/kev/runs/{RUN_ID}")
target = pathlib.Path(f"{DRIVE}/checkpoints/{RUN_ID}")
shutil.copytree(out, target, dirs_exist_ok=True)
shutil.copy(f"/content/train_{RUN_ID}.log", target / "train.log")
for name in ("adapter_model.safetensors", "head.pt"):
    print(name, hashlib.sha256((target / name).read_bytes()).hexdigest())
print(json.loads((target / "training_metrics.json").read_text())["grad_norm"])

After the run, on a machine with the repository: download `checkpoints/<RUN_ID>` from Drive,
score it with `analyst.evaluation.decision_eval`, then add it to the registry:

```bash
python -m analyst.models.kev_runs register models/checkpoints/<RUN_ID> --run-id <RUN_ID> \
    --drive-path Group4/checkpoints/<RUN_ID> --scores outputs/reports/score_domain_<RUN_ID>.json
```